# Benchmark inverse optimizers

Compare Adam, SGD, RMSprop and LBFGS on **the same 100 distinct, seeded, held-out scenarios**, then extend the comparison to Adam, LBFGS, PSO, differential evolution (DE), and PSO followed by LBFGS. Run from `notebooks/` after notebooks 01 and 02. The five-method report compares each neural recommendation with an independently optimized original steady-state plant. All methods use the same training-derived rate-feasible control domain and objective weights. Steps/generations have different evaluation costs; read both evaluations and elapsed time. Predicted output limits are soft penalties; the static surrogate does not capture lag.

In [ ]:
from pathlib import Path
from time import perf_counter
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, str(Path('../src').resolve()))
from constraints import CONTROL_NAMES, SigmoidControlParameterization
from objective_functions import InverseProcessObjective

SEED = 42
N_SCENARIOS = 100
MAX_STEPS = 60
FEASIBILITY_TOL = 1e-5
LEARNING_RATES = {'Adam': 0.06, 'SGD': 0.4, 'RMSprop': 0.03, 'LBFGS': 0.8}
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DATA_PATH = Path('../data/synthetic_process_data.csv')
MODEL_DIR = Path('../models')
RESULTS_DIR = Path('../results')
checkpoint = torch.load(MODEL_DIR / 'process_model.pt', map_location='cpu', weights_only=True)
# Pickle scalers must be trusted, locally generated artifacts from notebook 02.
with (MODEL_DIR / 'input_scaler.pkl').open('rb') as file:
    input_scaler = pickle.load(file)
with (MODEL_DIR / 'output_scaler.pkl').open('rb') as file:
    output_scaler = pickle.load(file)
input_columns, output_columns = checkpoint['input_columns'], checkpoint['output_columns']
if (len(input_columns) != 10 or input_columns[5:] != list(CONTROL_NAMES)
        or output_columns != ['y1_quality', 'y2_energy_load']
        or tuple(checkpoint['hidden_sizes']) != (32, 32, 16)):
    raise ValueError('Unexpected saved model dimensions or column order')
if (list(input_scaler.feature_names_in_) != input_columns
        or list(output_scaler.feature_names_in_) != output_columns):
    raise ValueError('Saved scalers and model have different feature order')

class ProcessModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 32), nn.SiLU(), nn.Linear(32, 32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU(), nn.Linear(16, 2),
        )

    def forward(self, features):
        return self.network(features)

model = ProcessModel()
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval().requires_grad_(False)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Generate data with notebook 01: {DATA_PATH.resolve()}')
data = pd.read_csv(DATA_PATH)
if len(data) != 8000 or not np.isfinite(data[input_columns + output_columns].to_numpy(dtype=float)).all():
    raise ValueError('Expected 8,000 finite observations in the generated dataset')
train, test = data.iloc[:int(0.7 * len(data))], data.iloc[int(0.85 * len(data)):]
if (not np.allclose(input_scaler.mean_, train[input_columns].mean().to_numpy(), rtol=1e-5)
        or not np.allclose(output_scaler.mean_, train[output_columns].mean().to_numpy(), rtol=1e-5)):
    raise ValueError('Scalers must be fitted to the chronological training split')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
support_lower = train[input_columns].quantile(0.01).to_numpy(dtype=np.float32)
support_upper = train[input_columns].quantile(0.99).to_numpy(dtype=np.float32)
origin = torch.from_numpy(support_lower[5:].copy())
span = torch.from_numpy((support_upper[5:] - support_lower[5:]).copy())
parameterization = SigmoidControlParameterization()
test_inputs = test[input_columns].to_numpy(dtype=np.float32)
test_commands = (test_inputs[:, 5:] - origin.numpy()) / span.numpy()
# Interior points make zero logits map exactly to the starting controls.
eligible = np.flatnonzero(
    np.all((test_inputs[:, :5] >= support_lower[:5]) & (test_inputs[:, :5] <= support_upper[:5]), axis=1)
    & np.all((test_commands >= parameterization.lower.numpy() + parameterization.max_move)
             & (test_commands <= parameterization.upper.numpy() - parameterization.max_move), axis=1)
)
if len(eligible) < N_SCENARIOS:
    raise ValueError(f'Only {len(eligible)} feasible interior test points; need {N_SCENARIOS}')
selected = rng.choice(eligible, size=N_SCENARIOS, replace=False)
scenario_indices = test.index[selected].to_numpy()
print(f'Benchmarking {len(selected)} distinct test points (from {len(eligible)} eligible)')

input_mean = torch.as_tensor(input_scaler.mean_, dtype=torch.float32)
input_scale = torch.as_tensor(input_scaler.scale_, dtype=torch.float32)
output_mean = torch.as_tensor(output_scaler.mean_, dtype=torch.float32)
output_scale = torch.as_tensor(output_scaler.scale_, dtype=torch.float32)
control_lower = origin + span * parameterization.lower
control_upper = origin + span * parameterization.upper
support_lower_tensor = torch.from_numpy(support_lower)
support_upper_tensor = torch.from_numpy(support_upper)

def predict(disturbances, controls):
    scaled = (torch.cat((disturbances, controls)) - input_mean) / input_scale
    return model(scaled.unsqueeze(0)).squeeze(0) * output_scale + output_mean

def make_objective(start_outputs):
    return InverseProcessObjective(
        model, input_mean, input_scale, output_mean, output_scale,
        control_lower, control_upper, support_lower_tensor, support_upper_tensor,
        quality_min=float(start_outputs[0] - 0.25),
        energy_max=float(start_outputs[1] + 0.25),
        quality_weight=1.0, energy_weight=1.0,
        movement_weight=0.1, constraint_weight=20.0, support_weight=10.0,
    )


In [ ]:
def make_optimizer(name, logits):
    rate = LEARNING_RATES[name]
    if name == 'Adam':
        return torch.optim.Adam([logits], lr=rate)
    if name == 'SGD':
        return torch.optim.SGD([logits], lr=rate)
    if name == 'RMSprop':
        return torch.optim.RMSprop([logits], lr=rate)
    if name == 'LBFGS':
        return torch.optim.LBFGS([logits], lr=rate, max_iter=1, max_eval=10,
                                 line_search_fn='strong_wolfe')
    raise ValueError(f'Unknown optimizer: {name}')

def evaluate(objective, disturbances, starting_controls, previous_commands, logits):
    with torch.no_grad():
        commands = parameterization(logits, previous_commands)
        controls = origin + span * commands
        outputs = predict(disturbances, controls)
        loss = objective(disturbances, controls, starting_controls)
        full = torch.cat((disturbances, controls))
        support_violation = torch.maximum(
            torch.relu((support_lower_tensor - full) / input_scale),
            torch.relu((full - support_upper_tensor) / input_scale),
        ).max().item()
        return {
            'objective': loss.item(), 'quality': outputs[0].item(), 'energy': outputs[1].item(),
            'quality_shortfall': max(objective.quality_min - outputs[0].item(), 0.0),
            'energy_excess': max(outputs[1].item() - objective.energy_max, 0.0),
            'bound_violation': torch.maximum(torch.relu(parameterization.lower - commands),
                                              torch.relu(commands - parameterization.upper)).max().item(),
            'rate_violation': torch.relu((commands - previous_commands).abs()
                                          - parameterization.max_move).max().item(),
            'support_violation': support_violation,
            'command_distance': torch.linalg.vector_norm(commands - previous_commands).item(),
            'physical_distance': torch.linalg.vector_norm(controls - starting_controls).item(),
            'controls': controls.clone(),
        }

runs, traces = [], []
for scenario_id, selected_row in enumerate(selected):
    disturbances = torch.from_numpy(test_inputs[selected_row, :5].copy())
    starting_controls = torch.from_numpy(test_inputs[selected_row, 5:].copy())
    previous_commands = torch.from_numpy(test_commands[selected_row].copy())
    with torch.no_grad():
        starting_outputs = predict(disturbances, starting_controls)
    objective = make_objective(starting_outputs)
    baseline = objective(disturbances, starting_controls, starting_controls).item()
    for name in LEARNING_RATES:
        logits = nn.Parameter(torch.zeros(5))
        optimizer = make_optimizer(name, logits)
        evaluations = 0
        elapsed_start = perf_counter()
        for step in range(1, MAX_STEPS + 1):
            closure_calls = [0]
            def closure():
                closure_calls[0] += 1
                optimizer.zero_grad(set_to_none=True)
                controls = origin + span * parameterization(logits, previous_commands)
                loss = objective(disturbances, controls, starting_controls)
                if not torch.isfinite(loss):
                    raise ValueError(f'{name} produced a non-finite loss for test sample {scenario_indices[scenario_id]}')
                loss.backward()
                if not torch.isfinite(logits.grad).all():
                    raise ValueError(f'{name} produced a non-finite gradient for test sample {scenario_indices[scenario_id]}')
                return loss

            if name == 'LBFGS':
                optimizer.step(closure)
            else:
                closure()
                optimizer.step()
            evaluations += closure_calls[0]
            state = evaluate(objective, disturbances, starting_controls, previous_commands, logits)
            traces.append({'scenario_id': scenario_id, 'test_index': scenario_indices[scenario_id],
                           'optimizer': name, 'step': step, 'evaluations': evaluations,
                           'objective': state['objective'],
                           'objective_gain': baseline - state['objective'],
                           'quality': state['quality'], 'energy': state['energy'],
                           'quality_shortfall': state['quality_shortfall'],
                           'energy_excess': state['energy_excess'],
                           'bound_violation': state['bound_violation'],
                           'rate_violation': state['rate_violation'],
                           'support_violation': state['support_violation'],
                           'command_distance': state['command_distance'],
                           'physical_distance': state['physical_distance']})
        solve_seconds = perf_counter() - elapsed_start
        violations = [state[key] for key in ('quality_shortfall', 'energy_excess',
                                               'bound_violation', 'rate_violation', 'support_violation')]
        runs.append({
            'scenario_id': scenario_id, 'test_index': scenario_indices[scenario_id], 'optimizer': name,
            'initial_objective': baseline, 'objective': state['objective'],
            'objective_gain': baseline - state['objective'],
            'initial_quality': starting_outputs[0].item(), 'quality': state['quality'],
            'quality_change': state['quality'] - starting_outputs[0].item(),
            'initial_energy': starting_outputs[1].item(), 'energy': state['energy'],
            'energy_change': state['energy'] - starting_outputs[1].item(),
            'quality_shortfall': state['quality_shortfall'], 'energy_excess': state['energy_excess'],
            'bound_violation': state['bound_violation'], 'rate_violation': state['rate_violation'],
            'support_violation': state['support_violation'], 'max_violation': max(violations),
            'feasible': max(violations) <= FEASIBILITY_TOL,
            'solve_seconds': solve_seconds, 'iterations': MAX_STEPS,
            'evaluations': evaluations, 'command_distance': state['command_distance'],
            'physical_distance': state['physical_distance'],
            **{f'optimal_{feature}': state['controls'][j].item()
               for j, feature in enumerate(CONTROL_NAMES)},
        })
runs_df = pd.DataFrame(runs)
traces_df = pd.DataFrame(traces)
assert len(runs_df) == N_SCENARIOS * len(LEARNING_RATES)
assert len(traces_df) == len(runs_df) * MAX_STEPS
runs_df.to_csv(RESULTS_DIR / 'optimizer_benchmark_runs.csv', index=False)
traces_df.to_csv(RESULTS_DIR / 'optimizer_benchmark_convergence.csv', index=False)
print(f'Exported {len(runs_df)} runs and {len(traces_df)} convergence records')


## Summary and balance ranking

Performance is improvement in the *same* penalized objective versus each scenario's own start (larger gain is better). Stability is the 10th percentile objective gain and fraction of scenarios with positive gain; constraint satisfaction is the fraction with every violation at most 1e-5; speed is median wall-clock solve time. To make the decision reproducible, rank each optimizer on those five metrics (rank 1 = best), sum ranks equally, and break ties by feasibility, median gain, then speed. These are illustrative trade-off weights, not universal process-control guarantees.

In [ ]:
summary = runs_df.groupby('optimizer', sort=False).agg(
    scenarios=('scenario_id', 'count'),
    median_objective=('objective', 'median'),
    median_objective_gain=('objective_gain', 'median'),
    p10_objective_gain=('objective_gain', lambda x: x.quantile(0.10)),
    improvement_rate=('objective_gain', lambda x: (x > 0).mean()),
    median_quality=('quality', 'median'),
    median_energy=('energy', 'median'),
    feasibility_rate=('feasible', 'mean'),
    worst_violation=('max_violation', 'max'),
    median_solve_seconds=('solve_seconds', 'median'),
    median_iterations=('iterations', 'median'),
    median_evaluations=('evaluations', 'median'),
    median_command_distance=('command_distance', 'median'),
    median_physical_distance=('physical_distance', 'median'),
).reset_index()
directions = {'feasibility_rate': False, 'median_objective_gain': False,
              'p10_objective_gain': False, 'improvement_rate': False,
              'median_solve_seconds': True}
for metric, ascending in directions.items():
    summary[f'rank_{metric}'] = summary[metric].rank(ascending=ascending, method='min')
summary['balance_rank_sum'] = summary[[f'rank_{metric}' for metric in directions]].sum(axis=1)
summary = summary.sort_values(['balance_rank_sum', 'feasibility_rate',
                               'median_objective_gain', 'median_solve_seconds', 'optimizer'],
                              ascending=[True, False, False, True, True]).reset_index(drop=True)
summary['balance_rank'] = np.arange(1, len(summary) + 1)
summary.to_csv(RESULTS_DIR / 'optimizer_benchmark_summary.csv', index=False)
display(summary)
winner = summary.iloc[0]
print(f'Best balanced optimizer under the stated equal-rank rule: {winner["optimizer"]} '
      f'(feasible {winner["feasibility_rate"]:.0%}, median objective gain '
      f'{winner["median_objective_gain"]:.4f}, p10 gain {winner["p10_objective_gain"]:.4f}, '
      f'median time {winner["median_solve_seconds"]:.4f}s)')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax, column, label in zip(axes.flat,
                             ['objective_gain', 'max_violation', 'solve_seconds', 'command_distance'],
                             ['Objective gain vs own start', 'Maximum constraint violation',
                              'Solve time (seconds)', 'Distance from start (command L2)']):
    values = [runs_df.loc[runs_df.optimizer == name, column].to_numpy()
              for name in LEARNING_RATES]
    ax.boxplot(values, tick_labels=list(LEARNING_RATES), showfliers=False)
    ax.set(ylabel=label, title=label)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('100 paired inverse-optimization scenarios')
fig.savefig(RESULTS_DIR / 'optimizer_benchmark_boxplots.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for name in LEARNING_RATES:
    trace = traces_df.loc[traces_df.optimizer == name]
    for ax, column in zip(axes, ['objective_gain', 'objective']):
        grouped = trace.groupby('step')[column]
        center = grouped.median()
        lo = grouped.quantile(0.1)
        hi = grouped.quantile(0.9)
        line, = ax.plot(center.index, center.values, label=name)
        ax.fill_between(center.index, lo.values, hi.values, color=line.get_color(), alpha=0.12)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set(xlabel='Optimizer step', ylabel='Gain vs start', title='Median objective gain (10th–90th percentile)')
axes[1].set(xlabel='Optimizer step', ylabel='Objective loss', title='Median objective (10th–90th percentile)')
axes[0].legend()
fig.savefig(RESULTS_DIR / 'optimizer_benchmark_convergence.png', dpi=150)
plt.show()


## Five-method comparison against the original process

For each of the **same 100 scenarios**: optimize the neural objective, then independently find an **approximate true optimum** by two seeded DE searches and bounded local polishing of the original noise-free `steady_state_response` from notebook 01. The oracle uses the *same starting controls, predicted-start-based quality/energy limits, scaling, movement, support and violation penalties* as the neural objective. Both searches use the identical rate-feasible command box. The oracle is a numerical reference, **not a certified global optimum**. We record predicted and true outputs at both recommendations. Optimization error is each method's predicted objective minus the best predicted objective found by the five methods on that scenario (nonnegative). True regret is the true objective minus the numerical oracle's true objective. Generalization error is mean absolute standardized output prediction error, both at the held-out operating point and at the recommended controls; the latter exposes distribution shift. Model exploitation is *predicted improvement minus true improvement* under matched objectives, so positive values mean optimistic model-reported gains. Constraint satisfaction uses actual plant outputs and the common limits; negative true regret caused by an imperfect oracle is reported, not hidden.

In [ ]:
import ast
import nbformat
from scipy.optimize import differential_evolution, minimize
from scipy.special import expit

plant_path = Path('01_generate_data.ipynb')
plant_notebook = nbformat.read(plant_path, as_version=4)
sources = [ast.parse(cell.source) for cell in plant_notebook.cells
           if cell.cell_type == 'code' and 'def steady_state_response(' in cell.source]
if len(sources) != 1:
    raise ValueError('Expected one original plant definition')
plant_nodes = [node for node in sources[0].body
               if isinstance(node, ast.FunctionDef) and node.name == 'steady_state_response']
if len(plant_nodes) != 1:
    raise ValueError('Original plant function not found')
plant_scope = {'np': np, 'expit': expit}
exec(compile(ast.Module(body=plant_nodes, type_ignores=[]), str(plant_path), 'exec'), plant_scope)
plant = plant_scope['steady_state_response']
METHODS = ('Adam', 'LBFGS', 'PSO', 'DE', 'PSO+LBFGS')
SWARM_SIZE = 20
GLOBAL_GENERATIONS = 60
rng_global = np.random.default_rng(SEED + 17)

def scenario_domain(previous_commands):
    lo = torch.maximum(parameterization.lower, previous_commands - parameterization.max_move).numpy()
    hi = torch.minimum(parameterization.upper, previous_commands + parameterization.max_move).numpy()
    if np.any(lo >= hi):
        raise ValueError('Empty rate-feasible domain')
    return lo, hi

def evaluate_neural(commands, disturbances, previous_controls, objective):
    """Evaluate a batch of feasible normalized commands with the exact surrogate objective."""
    values = np.atleast_2d(np.asarray(commands, dtype=np.float32))
    with torch.no_grad():
        controls = origin + span * torch.from_numpy(values)
        physical = torch.cat((disturbances.expand(len(values), -1), controls), dim=1)
        scaled = (physical - input_mean) / input_scale
        scaled_out = model(scaled)
        outputs = scaled_out * output_scale + output_mean
        move = 0.1 * (((controls - previous_controls) / input_scale[5:]) ** 2).sum(1)
        bound = 20 * (torch.relu((control_lower - controls) / input_scale[5:]).square()
                      + torch.relu((controls - control_upper) / input_scale[5:]).square()).sum(1)
        limits = 20 * (torch.relu((objective.quality_min - outputs[:, 0]) / output_scale[0]).square()
                       + torch.relu((outputs[:, 1] - objective.energy_max) / output_scale[1]).square())
        support = 10 * (torch.relu((support_lower_tensor - physical) / input_scale).square()
                        + torch.relu((physical - support_upper_tensor) / input_scale).square()).sum(1)
        loss = -scaled_out[:, 0] + scaled_out[:, 1] + move + bound + limits + support
    return loss.numpy().astype(np.float64)

def evaluate_plant(commands, disturbances, previous_controls, objective):
    """Matched scalar objective with true noise-free steady-state outputs."""
    values = np.atleast_2d(np.asarray(commands, dtype=np.float64))
    controls = origin.numpy() + span.numpy() * values
    physical = np.column_stack((np.broadcast_to(disturbances.numpy(), (len(values), 5)), controls))
    outputs = plant(physical[:, :5], controls)
    scaled_out = (outputs - output_mean.numpy()) / output_scale.numpy()
    move = 0.1 * np.sum(((controls - previous_controls.numpy()) / input_scale.numpy()[5:]) ** 2, axis=1)
    bound = 20 * np.sum(np.maximum((control_lower.numpy() - controls) / input_scale.numpy()[5:], 0) ** 2
                        + np.maximum((controls - control_upper.numpy()) / input_scale.numpy()[5:], 0) ** 2, axis=1)
    limits = 20 * (np.maximum((objective.quality_min - outputs[:, 0]) / output_scale.numpy()[0], 0) ** 2
                   + np.maximum((outputs[:, 1] - objective.energy_max) / output_scale.numpy()[1], 0) ** 2)
    support = 10 * np.sum(np.maximum((support_lower - physical) / input_scale.numpy(), 0) ** 2
                          + np.maximum((physical - support_upper) / input_scale.numpy(), 0) ** 2, axis=1)
    return -scaled_out[:, 0] + scaled_out[:, 1] + move + bound + limits + support, outputs

def swarm_search(score, lo, hi, seed, generations, size=SWARM_SIZE):
    random = np.random.default_rng(seed)
    positions = random.uniform(lo, hi, (size, 5))
    positions[0] = (lo + hi) / 2
    velocities = np.zeros_like(positions)
    scores = score(positions)
    best_positions = positions.copy()
    best_scores = scores.copy()
    winner = best_scores.argmin()
    global_best = best_positions[winner].copy()
    global_score = best_scores[winner]
    trace = []
    for generation in range(1, generations + 1):
        velocities = (0.65 * velocities + 1.4 * random.random(positions.shape) * (best_positions - positions)
                      + 1.4 * random.random(positions.shape) * (global_best - positions))
        positions = np.clip(positions + velocities, lo, hi)
        scores = score(positions)
        improved = scores < best_scores
        best_positions[improved] = positions[improved]
        best_scores[improved] = scores[improved]
        winner = best_scores.argmin()
        if best_scores[winner] < global_score:
            global_score = best_scores[winner]
            global_best = best_positions[winner].copy()
        trace.append(global_score)
    return global_best, np.array(trace), size * (generations + 1)

def bounded_de(score, lo, hi, seed, generations):
    trace = []
    def objective_vector(values):
        return score(values.T)
    def callback(intermediate_result):
        trace.append(float(intermediate_result.fun))
    result = differential_evolution(objective_vector, list(zip(lo, hi)),
                                    popsize=4, maxiter=generations, tol=0, atol=0,
                                    seed=seed, vectorized=True, polish=False,
                                    callback=callback)
    if not np.isfinite(result.fun) or len(trace) != generations:
        raise ValueError('DE did not produce a finite full-length trajectory')
    return result.x.copy(), np.array(trace), result.nfev * 20

def refine_lbfgs(start, lo, hi, disturbances, previous_controls, objective, steps,
                 previous_commands=None):
    # Invert the feasible command map to initialize logits at a PSO solution.
    fraction = np.clip((start - lo) / (hi - lo), 1e-5, 1 - 1e-5)
    logits = nn.Parameter(torch.from_numpy(np.log(fraction / (1 - fraction)).astype(np.float32)))
    if previous_commands is None:
        previous_commands = (previous_controls - origin) / span
    solver = torch.optim.LBFGS([logits], lr=0.8, max_iter=1, max_eval=10,
                               line_search_fn='strong_wolfe')
    best = start.copy()
    best_loss = evaluate_neural(start, disturbances, previous_controls, objective)[0]
    trace = []
    calls = [0]
    for _ in range(steps):
        def closure():
            calls[0] += 1
            solver.zero_grad(set_to_none=True)
            controls = origin + span * parameterization(logits, previous_commands)
            loss = objective(disturbances, controls, previous_controls)
            if not torch.isfinite(loss):
                raise ValueError('Hybrid LBFGS produced non-finite objective')
            loss.backward()
            return loss
        solver.step(closure)
        with torch.no_grad():
            candidate = parameterization(logits, previous_commands).numpy().copy()
        value = evaluate_neural(candidate, disturbances, previous_controls, objective)[0]
        if value < best_loss:
            best, best_loss = candidate, value
        trace.append(best_loss)
    return best, np.array(trace), calls[0]


In [ ]:
extended_rows, oracle_rows, extended_traces = [], [], []
for scenario_id, row_index in enumerate(selected):
    disturbances = torch.from_numpy(test_inputs[row_index, :5].copy())
    start_controls = torch.from_numpy(test_inputs[row_index, 5:].copy())
    start_commands = test_commands[row_index].astype(np.float64)
    lo, hi = scenario_domain(torch.from_numpy(start_commands.astype(np.float32)))
    with torch.no_grad():
        start_predicted = predict(disturbances, start_controls)
    obj = make_objective(start_predicted)
    neural_score = lambda values: evaluate_neural(values, disturbances, start_controls, obj)
    true_score = lambda values: evaluate_plant(values, disturbances, start_controls, obj)[0]
    initial_predicted_loss = obj(disturbances, start_controls, start_controls).item()
    initial_true_loss, initial_true_outputs = evaluate_plant(start_commands, disturbances, start_controls, obj)
    checked = neural_score(start_commands)[0]
    if not np.isclose(checked, initial_predicted_loss, atol=1e-5):
        raise ValueError('Batched surrogate score disagrees with differentiable objective')
    oracle_candidates = []
    for attempt in range(2):
        candidate, _, _ = bounded_de(true_score, lo, hi, SEED + scenario_id * 2 + attempt, 45)
        polished = minimize(lambda z: true_score(z)[0], candidate, method='L-BFGS-B',
                            bounds=list(zip(lo, hi)), options={'maxiter': 80, 'ftol': 1e-11})
        oracle_candidates.extend((candidate, polished.x))
    oracle_candidates.append(start_commands)
    oracle_values = true_score(np.array(oracle_candidates))
    oracle_command = np.array(oracle_candidates)[np.argmin(oracle_values)]
    oracle_loss, oracle_outputs = evaluate_plant(oracle_command, disturbances, start_controls, obj)
    oracle_prediction = predict(disturbances, origin + span * torch.tensor(oracle_command, dtype=torch.float32)).detach().numpy()
    oracle_quality_shortfall = max(obj.quality_min - oracle_outputs[0, 0], 0)
    oracle_energy_excess = max(oracle_outputs[0, 1] - obj.energy_max, 0)
    oracle_rows.append({'scenario_id': scenario_id, 'test_index': int(scenario_indices[scenario_id]),
                        'true_optimum_objective': oracle_loss[0],
                        'true_optimum_quality': oracle_outputs[0, 0],
                        'true_optimum_energy': oracle_outputs[0, 1],
                        'true_optimum_quality_shortfall': oracle_quality_shortfall,
                        'true_optimum_energy_excess': oracle_energy_excess,
                        'true_optimum_output_feasible': max(oracle_quality_shortfall, oracle_energy_excess) <= FEASIBILITY_TOL,
                        'predicted_at_true_optimum_quality': oracle_prediction[0],
                        'predicted_at_true_optimum_energy': oracle_prediction[1],
                        **{f'true_optimum_{feature}': (origin.numpy() + span.numpy() * oracle_command)[j]
                           for j, feature in enumerate(CONTROL_NAMES)}})
    candidates = {}
    for name in ('Adam', 'LBFGS'):
        prior = runs_df[(runs_df.scenario_id == scenario_id) & (runs_df.optimizer == name)].iloc[0]
        commands = (prior[[f'optimal_{feature}' for feature in CONTROL_NAMES]].to_numpy(dtype=float)
                    - origin.numpy()) / span.numpy()
        candidates[name] = (commands, float(prior.solve_seconds), int(prior.evaluations))
    started = perf_counter()
    pso_command, pso_trace, pso_evaluations = swarm_search(
        neural_score, lo, hi, SEED + scenario_id, GLOBAL_GENERATIONS)
    pso_time = perf_counter() - started
    candidates['PSO'] = (pso_command, pso_time, pso_evaluations)
    started = perf_counter()
    de_command, de_trace, de_evaluations = bounded_de(
        neural_score, lo, hi, SEED + scenario_id, GLOBAL_GENERATIONS)
    de_time = perf_counter() - started
    candidates['DE'] = (de_command, de_time, de_evaluations)
    started = perf_counter()
    hybrid_seed, hybrid_pso_trace, hybrid_pso_evaluations = swarm_search(
        neural_score, lo, hi, SEED + 10000 + scenario_id, 40)
    hybrid_command, hybrid_lbfgs_trace, hybrid_lbfgs_evaluations = refine_lbfgs(
        hybrid_seed, lo, hi, disturbances, start_controls, obj, 20)
    hybrid_time = perf_counter() - started
    candidates['PSO+LBFGS'] = (hybrid_command, hybrid_time,
                               hybrid_pso_evaluations + hybrid_lbfgs_evaluations)
    traces = {'PSO': pso_trace, 'DE': de_trace,
              'PSO+LBFGS': np.r_[hybrid_pso_trace, hybrid_lbfgs_trace]}
    for name in ('Adam', 'LBFGS'):
        traces[name] = traces_df.loc[(traces_df.scenario_id == scenario_id)
                                     & (traces_df.optimizer == name), 'objective'].to_numpy()
    for name in METHODS:
        command, seconds, evaluations = candidates[name]
        if np.any(command < lo - 1e-5) or np.any(command > hi + 1e-5):
            raise ValueError(f'{name} violated the rate-feasible command domain')
        pred_loss = neural_score(command)[0]
        true_loss, true_outputs = evaluate_plant(command, disturbances, start_controls, obj)
        controls = origin.numpy() + span.numpy() * command
        predicted = predict(disturbances, torch.tensor(controls, dtype=torch.float32)).detach().numpy()
        pred_gain = initial_predicted_loss - pred_loss
        true_gain = initial_true_loss[0] - true_loss[0]
        true_violations = [max(obj.quality_min - true_outputs[0, 0], 0),
                           max(true_outputs[0, 1] - obj.energy_max, 0),
                           np.maximum(support_lower - np.r_[disturbances.numpy(), controls], 0).max(),
                           np.maximum(np.r_[disturbances.numpy(), controls] - support_upper, 0).max(),
                           np.maximum(parameterization.lower.numpy() - command, 0).max(),
                           np.maximum(command - parameterization.upper.numpy(), 0).max(),
                           np.maximum(np.abs(command - start_commands) - parameterization.max_move, 0).max()]
        extended_rows.append({
            'scenario_id': scenario_id, 'test_index': int(scenario_indices[scenario_id]), 'optimizer': name,
            'predicted_objective': pred_loss, 'true_objective': true_loss[0],
            'predicted_quality': predicted[0], 'predicted_energy': predicted[1],
            'true_quality': true_outputs[0, 0], 'true_energy': true_outputs[0, 1],
            'predicted_gain': pred_gain, 'true_gain': true_gain,
            'model_exploitation': pred_gain - true_gain,
            'heldout_start_generalization_error': np.mean(np.abs(
                (start_predicted.numpy() - initial_true_outputs[0]) / output_scale.numpy())),
            'optimized_generalization_error': np.mean(np.abs(
                (predicted - true_outputs[0]) / output_scale.numpy())),
            'true_regret': true_loss[0] - oracle_loss[0],
            'max_true_violation': max(true_violations),
            'true_feasible': max(true_violations) <= FEASIBILITY_TOL,
            'quality_shortfall_true': true_violations[0], 'energy_excess_true': true_violations[1],
            'solve_seconds': seconds, 'evaluations': evaluations,
            'command_distance': np.linalg.norm(command - start_commands),
            **{f'predicted_optimum_{feature}': controls[j]
               for j, feature in enumerate(CONTROL_NAMES)},
        })
        extended_traces.extend({'scenario_id': scenario_id, 'optimizer': name,
                                'step': step, 'predicted_objective': value}
                               for step, value in enumerate(traces[name], start=1))
extended = pd.DataFrame(extended_rows)
oracles = pd.DataFrame(oracle_rows)
extended['optimization_error'] = extended['predicted_objective'] - extended.groupby(
    'scenario_id')['predicted_objective'].transform('min')
extended.to_csv(RESULTS_DIR / 'optimizer_control_runs.csv', index=False)
oracles.to_csv(RESULTS_DIR / 'optimizer_true_optima.csv', index=False)
pd.DataFrame(extended_traces).to_csv(RESULTS_DIR / 'optimizer_control_convergence.csv', index=False)
print(f'Plant-aware benchmark: {len(extended)} method/scenario records and {len(oracles)} true references')


In [ ]:
control_summary = extended.groupby('optimizer', sort=False).agg(
    scenarios=('scenario_id', 'nunique'),
    median_true_regret=('true_regret', 'median'),
    p90_true_regret=('true_regret', lambda x: x.quantile(0.9)),
    median_optimization_error=('optimization_error', 'median'),
    median_model_exploitation=('model_exploitation', 'median'),
    median_abs_model_exploitation=('model_exploitation', lambda x: x.abs().median()),
    p90_model_exploitation=('model_exploitation', lambda x: x.quantile(0.9)),
    max_model_exploitation=('model_exploitation', 'max'),
    median_optimized_generalization_error=('optimized_generalization_error', 'median'),
    median_start_generalization_error=('heldout_start_generalization_error', 'median'),
    true_feasibility_rate=('true_feasible', 'mean'),
    median_true_gain=('true_gain', 'median'),
    true_improvement_rate=('true_gain', lambda x: (x > 0).mean()),
    median_solve_seconds=('solve_seconds', 'median'),
    median_evaluations=('evaluations', 'median'),
).reset_index()
rank_metrics = {'true_feasibility_rate': False, 'median_true_regret': True,
                'p90_true_regret': True, 'median_abs_model_exploitation': True,
                'median_solve_seconds': True}
for column, ascending in rank_metrics.items():
    control_summary[f'rank_{column}'] = control_summary[column].rank(
        ascending=ascending, method='min')
control_summary['rank_sum'] = control_summary[[f'rank_{x}' for x in rank_metrics]].sum(axis=1)
control_summary = control_summary.sort_values(
    ['rank_sum', 'true_feasibility_rate', 'median_true_regret', 'median_solve_seconds', 'optimizer'],
    ascending=[True, False, True, True, True]).reset_index(drop=True)
control_summary.insert(0, 'rank', np.arange(1, len(control_summary) + 1))
control_summary.to_csv(RESULTS_DIR / 'optimizer_control_summary.csv', index=False)
display(control_summary[['rank', 'optimizer', 'scenarios', 'true_feasibility_rate',
                         'median_true_regret', 'p90_true_regret', 'median_optimization_error',
                         'median_model_exploitation', 'p90_model_exploitation',
                         'true_improvement_rate', 'median_optimized_generalization_error',
                         'median_solve_seconds', 'median_evaluations']])
winner = control_summary.iloc[0]
oracle_feasibility = oracles.true_optimum_output_feasible.mean()
report_lines = [
    '# Inverse optimization: process-control comparison', '',
    f'Rank 1: **{winner.optimizer}** under the five equally ranked criteria below.',
    'These scores are research diagnostics, **not a deployment recommendation**.', '',
    f'Numerical oracle output feasibility: {oracle_feasibility:.1%}; the remaining cases also have '
    'output violations at the minimum-penalty reference. This is not proof that no feasible point exists.',
    f'The top method is truly feasible in {winner.true_feasibility_rate:.1%} and improves the '
    f'original plant objective in {winner.true_improvement_rate:.1%} of scenarios. '
    'No method should be used for control without independent safety and closed-loop validation.', '',
    '## Ranking rule',
    'Rank 1 is best for true-plant feasibility rate, median and 90th-percentile true regret, '
    'median absolute model exploitation, and median solve time. Sum the five ranks; break ties '
    'by feasibility, regret, then time. Lower regret and error are better.', '',
    '## Results (100 paired held-out scenarios)', '',
    '| Rank | Method | Feasible | True improvement | Median true regret | P90 regret | Median optimization error | Median exploitation | P90 exploitation | Median optimized error | Median seconds |',
    '|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|',
]
for row in control_summary.itertuples():
    report_lines.append(f'| {row.rank} | {row.optimizer} | {row.true_feasibility_rate:.1%} | '
                        f'{row.true_improvement_rate:.1%} | '
                        f'{row.median_true_regret:.4f} | {row.p90_true_regret:.4f} | '
                        f'{row.median_optimization_error:.4f} | {row.median_model_exploitation:.4f} | '
                        f'{row.p90_model_exploitation:.4f} | '
                        f'{row.median_optimized_generalization_error:.4f} | '
                        f'{row.median_solve_seconds:.4f} |')
report_lines += ['', '## Interpretation',
    '- The oracle is the best of two seeded DE runs, bounded local polishing, and the starting point '
    'on the original noise-free steady-state plant; global optimality is not proven.',
    '- Optimization error compares predicted losses to the best found by the five methods, not a certified neural global optimum.',
    '- Exploitation = neural predicted gain minus actual steady-state gain under matched penalties; '
    'positive means the model overstated improvement.',
    '- Generalization error is mean absolute output prediction error scaled by training output standard deviations; '
    'the optimized error captures off-policy shift.',
    '- True feasibility includes actual output thresholds, support, actuator and rate limits. '
    'Output limits are soft in the objective; feasibility cannot be assumed.',
    '- Solve times exclude oracle computation and depend on the machine. Steps are not equal-cost: '
    'evaluate the number of model calls and tail regret as well.',
    '- The true oracle and comparisons are static steady-state counterfactuals. Confirm controller '
    'performance using the lagged closed-loop plant in notebook 06 before control use.', '',
    'See optimizer_control_runs.csv, optimizer_true_optima.csv, optimizer_control_summary.csv '
    'and optimizer_control_convergence.csv for full data.',
]
(RESULTS_DIR / 'optimizer_control_report.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, metric, title in zip(axes, ('true_regret', 'optimized_generalization_error'),
                             ('True-plant regret', 'Off-policy generalization error')):
    ax.boxplot([extended.loc[extended.optimizer == name, metric] for name in METHODS],
               tick_labels=METHODS, showfliers=False)
    ax.set(ylabel=title, title=title)
    ax.tick_params(axis='x', rotation=25)
fig.savefig(RESULTS_DIR / 'optimizer_control_diagnostics.png', dpi=150)
plt.show()
print(f'Report written: {(RESULTS_DIR / "optimizer_control_report.md").resolve()}')


## Full-dataset optimization quality check (all 8,000 rows)

The preceding deep benchmark uses 100 eligible held-out rows. This additional sweep checks **every row**, including 5,600 train, 1,200 validation and 1,200 test rows. When a historical control is outside the normalized actuator bounds, project it to the closest bound and record the original-to-projected adjustment; optimize relative to this feasible reference. Disturbances are never projected. Disturbance support violations are still reported. The full sweep uses 8 Adam/LBFGS steps, 8 PSO/DE generations (20 candidates), hybrid 5 PSO generations + 3 LBFGS steps, and a separate 12-generation DE plus local polish for the approximate true-process reference. This is a **screening** budget; its rankings must not be conflated with the deeper 100-row benchmark. Timings exclude oracle computation. The original process reference remains steady-state rather than dynamic.

In [ ]:
FULL_STEPS = 8
FULL_GLOBAL_GENERATIONS = 8
FULL_ORACLE_GENERATIONS = 12
full_inputs = data[input_columns].to_numpy(dtype=np.float32)
full_rows, full_oracles = [], []
command_lower = parameterization.lower.numpy()
command_upper = parameterization.upper.numpy()
for sample_index, features in enumerate(full_inputs):
    disturbances = torch.from_numpy(features[:5].copy())
    original_commands = (features[5:] - origin.numpy()) / span.numpy()
    start_commands = np.clip(original_commands, command_lower, command_upper).astype(np.float32)
    projected_controls = origin + span * torch.from_numpy(start_commands)
    previous_commands = torch.from_numpy(start_commands)
    lo, hi = scenario_domain(previous_commands)
    split = 'train' if sample_index < 5600 else 'validation' if sample_index < 6800 else 'test'
    with torch.no_grad():
        starting_prediction = predict(disturbances, projected_controls)
    objective = make_objective(starting_prediction)
    neural_score = lambda commands: evaluate_neural(commands, disturbances, projected_controls, objective)
    true_score = lambda commands: evaluate_plant(commands, disturbances, projected_controls, objective)[0]
    initial_predicted_loss = neural_score(start_commands)[0]
    initial_true_loss, initial_true_outputs = evaluate_plant(start_commands, disturbances, projected_controls, objective)
    seed = SEED + sample_index
    oracle_command, _, _ = bounded_de(true_score, lo, hi, seed + 20000, FULL_ORACLE_GENERATIONS)
    polished = minimize(lambda commands: true_score(commands)[0], oracle_command,
                        method='L-BFGS-B', bounds=list(zip(lo, hi)),
                        options={'maxiter': 30})
    references = np.stack((start_commands, oracle_command, polished.x))
    oracle_command = references[np.argmin(true_score(references))]
    oracle_loss, oracle_outputs = evaluate_plant(oracle_command, disturbances, projected_controls, objective)
    oracle_physical = origin.numpy() + span.numpy() * oracle_command
    full_oracles.append({
        'sample_index': sample_index, 'split': split, 'true_optimum_objective': oracle_loss[0],
        'true_optimum_quality': oracle_outputs[0, 0], 'true_optimum_energy': oracle_outputs[0, 1],
        'true_optimum_quality_shortfall': max(objective.quality_min - oracle_outputs[0, 0], 0),
        'true_optimum_energy_excess': max(oracle_outputs[0, 1] - objective.energy_max, 0),
        **{f'true_optimum_{feature}': oracle_physical[j] for j, feature in enumerate(CONTROL_NAMES)},
    })
    candidates = {}
    for name in ('Adam', 'LBFGS'):
        logits = nn.Parameter(torch.zeros(5))
        solver = (torch.optim.Adam([logits], lr=LEARNING_RATES['Adam']) if name == 'Adam' else
                  torch.optim.LBFGS([logits], lr=LEARNING_RATES['LBFGS'], max_iter=1,
                                    max_eval=10, line_search_fn='strong_wolfe'))
        calls = [0]
        started = perf_counter()
        for _ in range(FULL_STEPS):
            def closure():
                calls[0] += 1
                solver.zero_grad(set_to_none=True)
                controls = origin + span * parameterization(logits, previous_commands)
                loss = objective(disturbances, controls, projected_controls)
                if not torch.isfinite(loss):
                    raise ValueError(f'{name}: non-finite loss at row {sample_index}')
                loss.backward()
                if not torch.isfinite(logits.grad).all():
                    raise ValueError(f'{name}: non-finite gradient at row {sample_index}')
                return loss
            if name == 'Adam':
                closure()
                solver.step()
            else:
                solver.step(closure)
        seconds = perf_counter() - started
        with torch.no_grad():
            proposed = parameterization(logits, previous_commands).numpy().copy()
        candidates[name] = (proposed, seconds, calls[0])
    started = perf_counter()
    pso_command, _, pso_calls = swarm_search(neural_score, lo, hi, seed, FULL_GLOBAL_GENERATIONS)
    candidates['PSO'] = (pso_command, perf_counter() - started, pso_calls)
    started = perf_counter()
    de_command, _, de_calls = bounded_de(neural_score, lo, hi, seed, FULL_GLOBAL_GENERATIONS)
    candidates['DE'] = (de_command, perf_counter() - started, de_calls)
    started = perf_counter()
    hybrid_seed, _, hybrid_pso_calls = swarm_search(neural_score, lo, hi, seed + 10000, 5)
    hybrid_command, _, hybrid_lbfgs_calls = refine_lbfgs(
        hybrid_seed, lo, hi, disturbances, projected_controls, objective, 3,
        previous_commands=previous_commands)
    candidates['PSO+LBFGS'] = (hybrid_command, perf_counter() - started,
                               hybrid_pso_calls + hybrid_lbfgs_calls)
    for name, (candidate, seconds, calls) in candidates.items():
        # The projected starting command is always a feasible fallback.
        command = min((candidate, start_commands), key=lambda c: neural_score(c)[0])
        if np.any(command < lo - 1e-5) or np.any(command > hi + 1e-5):
            raise ValueError(f'{name}: control domain violation at row {sample_index}')
        controls = origin.numpy() + span.numpy() * command
        predicted = predict(disturbances, torch.tensor(controls, dtype=torch.float32)).detach().numpy()
        predicted_loss = neural_score(command)[0]
        true_loss, actual = evaluate_plant(command, disturbances, projected_controls, objective)
        full_vector = np.r_[disturbances.numpy(), controls]
        violations = {
            'quality_shortfall_true': max(objective.quality_min - actual[0, 0], 0),
            'energy_excess_true': max(actual[0, 1] - objective.energy_max, 0),
            'support_violation': max(np.maximum(support_lower - full_vector, 0).max(),
                                     np.maximum(full_vector - support_upper, 0).max()),
            'bound_violation': max(np.maximum(command_lower - command, 0).max(),
                                   np.maximum(command - command_upper, 0).max()),
            'rate_violation': np.maximum(np.abs(command - start_commands)
                                         - parameterization.max_move, 0).max(),
        }
        full_rows.append({
            'sample_index': sample_index, 'split': split, 'optimizer': name,
            'projection_distance': np.linalg.norm(original_commands - start_commands),
            'predicted_objective': predicted_loss, 'true_objective': true_loss[0],
            'predicted_quality': predicted[0], 'predicted_energy': predicted[1],
            'true_quality': actual[0, 0], 'true_energy': actual[0, 1],
            'predicted_gain': initial_predicted_loss - predicted_loss,
            'true_gain': initial_true_loss[0] - true_loss[0],
            'true_regret': true_loss[0] - oracle_loss[0],
            'start_prediction_error': np.mean(np.abs(
                (starting_prediction.numpy() - initial_true_outputs[0]) / output_scale.numpy())),
            'optimized_prediction_error': np.mean(np.abs(
                (predicted - actual[0]) / output_scale.numpy())),
            'quality_error': predicted[0] - actual[0, 0],
            'energy_error': predicted[1] - actual[0, 1],
            'distance_from_projected_start': np.linalg.norm(command - start_commands),
            'solve_seconds': seconds, 'evaluations': calls,
            **violations,
            **{f'original_{feature}': features[5 + j] for j, feature in enumerate(CONTROL_NAMES)},
            **{f'projected_{feature}': projected_controls[j].item() for j, feature in enumerate(CONTROL_NAMES)},
            **{f'optimal_{feature}': controls[j] for j, feature in enumerate(CONTROL_NAMES)},
        })
    if (sample_index + 1) % 500 == 0:
        print(f'Completed {sample_index + 1:,}/{len(data):,} rows', flush=True)
full_runs = pd.DataFrame(full_rows)
full_oracle = pd.DataFrame(full_oracles)
if len(full_runs) != len(data) * len(METHODS) or len(full_oracle) != len(data):
    raise ValueError('Full-dataset sweep did not cover every scenario')
full_runs['optimization_error'] = full_runs.predicted_objective - full_runs.groupby(
    'sample_index').predicted_objective.transform('min')
full_runs['model_exploitation'] = full_runs.predicted_gain - full_runs.true_gain
violation_columns = ['quality_shortfall_true', 'energy_excess_true',
                     'support_violation', 'bound_violation', 'rate_violation']
full_runs['constraint_satisfied'] = full_runs[violation_columns].max(axis=1) <= FEASIBILITY_TOL
full_runs.to_csv(RESULTS_DIR / 'optimizer_full_dataset_runs.csv', index=False)
full_oracle.to_csv(RESULTS_DIR / 'optimizer_full_dataset_true_optima.csv', index=False)
print(f'Completed {len(full_runs):,} optimization results and {len(full_oracle):,} plant references')


In [ ]:
quality_columns = ['sample_index', 'split', 'optimizer', 'predicted_quality',
                   'true_quality', 'quality_error', 'predicted_energy', 'true_energy',
                   'energy_error', 'optimization_error', 'true_regret',
                   'model_exploitation', 'start_prediction_error',
                   'optimized_prediction_error', 'quality_shortfall_true',
                   'energy_excess_true', 'support_violation', 'bound_violation',
                   'rate_violation', 'constraint_satisfied', 'projection_distance']
quality_matrix = full_runs[quality_columns].sort_values(['sample_index', 'optimizer'])
summary_matrix = full_runs.groupby(['split', 'optimizer'], sort=False).agg(
    rows=('sample_index', 'nunique'),
    mean_predicted_quality=('predicted_quality', 'mean'),
    mean_true_quality=('true_quality', 'mean'),
    quality_mae=('quality_error', lambda values: values.abs().mean()),
    mean_predicted_energy=('predicted_energy', 'mean'),
    mean_true_energy=('true_energy', 'mean'),
    energy_mae=('energy_error', lambda values: values.abs().mean()),
    median_optimization_error=('optimization_error', 'median'),
    median_true_regret=('true_regret', 'median'),
    p90_true_regret=('true_regret', lambda values: values.quantile(0.9)),
    median_exploitation=('model_exploitation', 'median'),
    p90_exploitation=('model_exploitation', lambda values: values.quantile(0.9)),
    mean_start_error=('start_prediction_error', 'mean'),
    mean_optimized_error=('optimized_prediction_error', 'mean'),
    constraint_satisfaction_rate=('constraint_satisfied', 'mean'),
    projected_start_rate=('projection_distance', lambda values: (values > 1e-6).mean()),
    improvement_rate=('true_gain', lambda values: (values > 0).mean()),
    median_solve_seconds=('solve_seconds', 'median'),
).reset_index()
summary_matrix.to_csv(RESULTS_DIR / 'optimizer_full_dataset_summary.csv', index=False)
quality_matrix.to_csv(RESULTS_DIR / 'optimizer_full_dataset_quality_matrix.csv', index=False)
violation_matrix = full_runs.pivot(index='sample_index', columns='optimizer', values=violation_columns)
violation_matrix.to_csv(RESULTS_DIR / 'optimizer_full_dataset_violation_matrix.csv')
oracle_matrix = full_oracle.sort_values('sample_index')
printout = '\n\n'.join((
    'FULL DATASET QUALITY SUMMARY MATRIX (5,600 train / 1,200 validation / 1,200 test):\n'
    + summary_matrix.to_string(index=False),
    'FULL DATASET QUALITY-CHECK MATRIX (every scenario and optimizer):\n'
    + quality_matrix.to_string(index=False),
    'FULL DATASET CONSTRAINT VIOLATION MATRIX (every scenario and optimizer):\n'
    + violation_matrix.to_string(),
    'FULL DATASET ORIGINAL-PLANT REFERENCE MATRIX (every scenario):\n'
    + oracle_matrix.to_string(index=False),
))
(RESULTS_DIR / 'optimizer_full_dataset_quality_printout.txt').write_text(printout + '\n', encoding='utf-8')
print(printout)
print('\nComplete matrices saved in results/optimizer_full_dataset_*.csv and *_quality_printout.txt')


In [ ]:
ranked = summary_matrix.copy()
criteria = {'constraint_satisfaction_rate': False, 'median_true_regret': True,
            'p90_true_regret': True, 'p90_exploitation': True,
            'median_solve_seconds': True}
for metric, ascending in criteria.items():
    ranked[f'rank_{metric}'] = ranked.groupby('split')[metric].rank(
        ascending=ascending, method='min')
ranked['rank_sum'] = ranked[[f'rank_{metric}' for metric in criteria]].sum(axis=1)
ranked = ranked.sort_values(['split', 'rank_sum', 'constraint_satisfaction_rate',
                             'median_true_regret', 'median_solve_seconds', 'optimizer'],
                            ascending=[True, True, False, True, True, True])
ranked['rank'] = ranked.groupby('split').cumcount() + 1
ranked.to_csv(RESULTS_DIR / 'optimizer_full_dataset_ranking.csv', index=False)
print('FULL DATASET RANK MATRIX (training, validation, test separately):')
print(ranked[['split', 'rank', 'optimizer', 'rows', 'constraint_satisfaction_rate',
              'median_true_regret', 'p90_true_regret', 'p90_exploitation',
              'median_solve_seconds', 'rank_sum']].to_string(index=False))
test_rank = ranked.loc[ranked.split == 'test'].sort_values('rank')
report = [
    '# Full-dataset process-control optimization check', '',
    'All 8,000 generated rows were checked: 5,600 training, 1,200 validation and 1,200 test.',
    'Starting controls outside actuator bounds were projected; their adjustment is reported per row.',
    'The full sweep uses a screening search budget (8 optimizer steps/generations and a 12-generation '
    'plant-reference DE search plus local polish), not the deeper 100-scenario budget.',
    'True optima are **approximate noise-free steady-state references**, not proven global optima.', '',
    '## Held-out test ranking (do not rank deployment decisions on training rows)', '',
    'Equal-rank sum: true constraint satisfaction (higher), median and 90th percentile true regret '
    '(lower), 90th percentile model exploitation (lower), and solve time (lower). '
    'Ties: feasibility, median regret, then time.', '',
    '| Rank | Method | Feasible | Median true regret | P90 true regret | P90 exploitation | Median seconds |',
    '|---:|---|---:|---:|---:|---:|---:|',
]
for row in test_rank.itertuples():
    report.append(f'| {row.rank} | {row.optimizer} | {row.constraint_satisfaction_rate:.1%} | '
                  f'{row.median_true_regret:.4f} | {row.p90_true_regret:.4f} | '
                  f'{row.p90_exploitation:.4f} | {row.median_solve_seconds:.4f} |')
report += ['', '## Quality checks', '',
    'The complete printed quality, violation, summary and original-plant reference matrices are '
    'in optimizer_full_dataset_quality_printout.txt; separate CSVs hold all rows for analysis.',
    'Positive exploitation means predicted gains exceeded actual steady-state gains. '
    'Generalization errors compare frozen-network outputs with the generating function, not '
    'the lagged and noisy historical measurements.',
    'Constraints include true quality/energy thresholds, disturbance and control training-support '
    'limits, normalized actuator bounds and the 0.05 move limit. Fixed out-of-support disturbances '
    'may make scenarios infeasible regardless of the optimizer.',
    'This is a static screening result; validate the ranking in the lagged closed-loop simulation '
    'before any process-control use. Training-row results are in-sample diagnostics.',
]
(RESULTS_DIR / 'optimizer_full_dataset_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')
print(f'Full-dataset report: {(RESULTS_DIR / "optimizer_full_dataset_report.md").resolve()}')
